In [3]:
import os
import sys
import pandas as pd

# 将项目根目录加入 Python 路径（解决 ModuleNotFoundError）
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 已添加项目根目录: {project_root}")
else:
    # 如果当前已经在根目录，直接添加当前目录
    if os.path.exists(os.path.join(current_dir, 'config.py')):
        sys.path.insert(0, current_dir)
        print(f"✅ 当前目录已是项目根目录: {current_dir}")
    else:
        print("⚠️ 请确保当前工作目录在项目根目录或 notebooks 文件夹下")

from config import get_driver
from src.retriever import find_matching_phages, find_similar_cases

print("✅ 所有模块导入成功")

✅ 已添加项目根目录: D:\IdeaProjects\vs\phage-workspace-mvp
✅ 所有模块导入成功


In [6]:
# 连接数据库
driver = get_driver()
print("✅ 数据库连接成功\n")

# 1. 节点统计
with driver.session() as session:
    result = session.run("""
        MATCH (n) 
        RETURN labels(n)[0] AS label, count(*) AS count
        ORDER BY count DESC
    """)
    print("📊 节点统计：")
    node_counts = {}
    for record in result:
        label = record['label']
        count = record['count']
        node_counts[label] = count
        print(f"   {label}: {count}")

# 2. 关系统计
with driver.session() as session:
    result2 = session.run("""
        MATCH ()-[r]->() 
        RETURN type(r) AS relationship, count(*) AS count
    """)
    print("\n🔗 关系统计：")
    for record in result2:
        print(f"   {record['relationship']}: {record['count']}")

# 3. 预期值与实际值对比（自动验证）
print("\n✅ 数据完整性自检：")
expected = {
    'Pathogen': 4,
    'ClinicalCase': 15,
    'Phage': 30,
    'PhageHostInteraction': 30
}
all_ok = True
for label, expected_count in expected.items():
    actual = node_counts.get(label, 0)
    status = "✅" if actual == expected_count else "❌"
    print(f"   {status} {label}: 预期 {expected_count}，实际 {actual}")
    if actual != expected_count:
        all_ok = False

if all_ok:
    print("🎉 所有节点数量符合预期！")
else:
    print("⚠️ 节点数量与预期不符，请检查导入是否完整。")

✅ 数据库连接成功

📊 节点统计：
   Phage: 30
   PhageHostInteraction: 30
   ClinicalCase: 15
   Pathogen: 4

🔗 关系统计：
   HAS_INTERACTION: 30
   TARGETS: 30
   INVOLVES_PATHOGEN: 15
   TREATED_WITH: 10

✅ 数据完整性自检：
   ✅ Pathogen: 预期 4，实际 4
   ✅ ClinicalCase: 预期 15，实际 15
   ✅ Phage: 预期 30，实际 30
   ✅ PhageHostInteraction: 预期 30，实际 30
🎉 所有节点数量符合预期！


In [8]:
# 查询 CASE-001 的所有属性
query = """
MATCH (c:ClinicalCase {case_id: 'CASE-001'})
OPTIONAL MATCH (c)-[:INVOLVES_PATHOGEN]->(p:Pathogen)
RETURN c.case_id AS case_id,
       c.infection_type AS infection_type,
       c.infection_site AS infection_site,
       c.specimen_type AS specimen_type,
       c.patient_age_group AS age_group,
       c.comorbidities AS comorbidities,
       c.prior_antibiotics AS prior_antibiotics,
       c.phage_treatment AS phage_treatment,
       c.clinical_outcome AS clinical_outcome,
       c.microbiological_outcome AS microbiological_outcome,
       c.curated_by AS curated_by,
       c.curation_date AS curation_date,
       p.species AS pathogen_species,
       p.resistance_mechanism AS resistance,
       p.verification_status AS verification_status
"""

with driver.session() as session:
    result = session.run(query)
    record = result.single()
    
if record:
    print("🩺 CASE-001（李桂东病例）完整信息：")
    print("=" * 60)
    for key, value in dict(record).items():
        print(f"   {key}: {value}")
    print("=" * 60)
    print("✅ CASE-001 所有属性已正确导入")
else:
    print("❌ 未找到 CASE-001，请检查数据导入")

🩺 CASE-001（李桂东病例）完整信息：
   case_id: CASE-001
   infection_type: UTI
   infection_site: Urine
   specimen_type: Urine
   age_group: 65-75
   comorbidities: ['Bladder stones', 'BPH', 'Rectal cancer post-op', 'Hypertension', 'Varicose veins']
   prior_antibiotics: ['Levofloxacin', 'Minocycline', 'Meropenem']
   phage_treatment: CP-p-BC-23086, CP-p-BC-23062
   clinical_outcome: Clinical improvement at Day 7
   microbiological_outcome: Bacteria decreased from >99999 to 14089 CFU/μL, urine culture negative at Day 5
   curated_by: FDE-01
   curation_date: 2026-05-20
   pathogen_species: Escherichia coli
   resistance: MDR
   verification_status: MICROBIOLOGY_LAB_VERIFIED
✅ CASE-001 所有属性已正确导入


In [9]:
# 查询 CASE-001 使用了哪些噬菌体（通过 TREATED_WITH 关系）
query = """
MATCH (c:ClinicalCase {case_id: 'CASE-001'})-[r:TREATED_WITH]->(ph:Phage)
RETURN ph.name AS phage_name, ph.phage_id AS phage_id
"""

with driver.session() as session:
    result = session.run(query)
    phages = [dict(record) for record in result]

if phages:
    print("🧬 CASE-001 使用的噬菌体（通过 TREATED_WITH 关系）：")
    for p in phages:
        print(f"   ✅ {p['phage_name']} ({p['phage_id']})")
else:
    print("⚠️ 未找到 TREATED_WITH 关系，请检查导入顺序（应先导入 Phage 再导入 Case）")

🧬 CASE-001 使用的噬菌体（通过 TREATED_WITH 关系）：
   ✅ CP-p-BC-23086 (PHAGE-001)
   ✅ CP-p-BC-23062 (PHAGE-002)


In [8]:
# 诊断：检查是否有任何 TREATED_WITH 关系
with driver.session() as session:
    result = session.run("MATCH ()-[r:TREATED_WITH]->() RETURN count(r) AS cnt")
    cnt = result.single()['cnt']
    print(f"数据库中 TREATED_WITH 关系总数：{cnt}")

数据库中 TREATED_WITH 关系总数：2


In [10]:
# 查询 CP-p-BC-23086 的完整关系链
query = """
MATCH (ph:Phage {name: 'CP-p-BC-23086'})-[:HAS_INTERACTION]->(i:PhageHostInteraction)-[:TARGETS]->(p:Pathogen)
RETURN ph.name AS phage_name,
       i.evidence_level AS evidence_level,
       i.infection_probability AS probability,
       i.evidence_ref AS evidence_ref,
       p.species AS target_species
"""

with driver.session() as session:
    result = session.run(query)
    record = result.single()

if record:
    print("🔬 CP-p-BC-23086 的完整关系链：")
    print("=" * 60)
    for key, value in dict(record).items():
        print(f"   {key}: {value}")
    print("=" * 60)
    print("✅ HAS_INTERACTION 和 TARGETS 关系已正确建立")
else:
    print("❌ 未找到 CP-p-BC-23086 的关系链，请检查 PhageHostInteraction 导入")

🔬 CP-p-BC-23086 的完整关系链：
   phage_name: CP-p-BC-23086
   evidence_level: L3
   probability: 0.98
   evidence_ref: ['CASE-001']
   target_species: Escherichia coli
✅ HAS_INTERACTION 和 TARGETS 关系已正确建立


In [11]:
# 调用 retriever 验证查询功能
# 测试 find_matching_phages
print("🔍 测试 find_matching_phages（E. coli MDR）：")
phages = find_matching_phages(driver, "Escherichia coli", "MDR", limit=5)
print(f"找到 {len(phages)} 个匹配噬菌体：")
for p in phages:
    print(f"   {p['name']} (L{p['evidence_level']}, {p['infection_probability']})")

print("\n" + "-" * 60)

# 测试 find_similar_cases
print("🔍 测试 find_similar_cases（E. coli UTI）：")
cases = find_similar_cases(driver, "Escherichia coli", "UTI", limit=5)
print(f"找到 {len(cases)} 个相似病例：")
for c in cases:
    phages_used = c.get('phages_used', [])
    print(f"   {c['case_id']}: 结局 {c['clinical_outcome']}, 噬菌体: {phages_used}")

🔍 测试 find_matching_phages（E. coli MDR）：
找到 5 个匹配噬菌体：
   CP-p-BC-23086 (LL3, 0.98)
   CP-p-BC-23062 (LL3, 0.96)
   vB_EcoM_006 (LL2, 0.89)
   vB_EcoM_003 (LL2, 0.88)
   vB_EcoM_012 (LL2, 0.79)

------------------------------------------------------------
🔍 测试 find_similar_cases（E. coli UTI）：
找到 3 个相似病例：
   CASE-006: 结局 Not improved, 噬菌体: []
   CASE-003: 结局 Improved, 噬菌体: []
   CASE-001: 结局 Clinical improvement at Day 7, 噬菌体: ['CP-p-BC-23086', 'CP-p-BC-23062']


In [9]:
# ========== V1 必填字段填充率验证 ==========
from config import get_driver   # 请确保 config 中有 get_driver() 方法

# 必填字段列表（所有字段都在一条查询中通过关系获取）
required_fields = [
    "case_id", "pathogen_id", "species", "resistance_mechanism",
    "infection_type", "infection_site", "specimen_type", "verification_status"
]

# 一条 Cypher 关联查询，同时取出 ClinicalCase 和 Pathogen 上的必填字段
query = """
MATCH (c:ClinicalCase)-[:INVOLVES_PATHOGEN]->(p:Pathogen)
RETURN c.case_id AS case_id,
       c.infection_type AS infection_type,
       c.infection_site AS infection_site,
       c.specimen_type AS specimen_type,
       p.pathogen_id AS pathogen_id,
       p.species AS species,
       p.resistance_mechanism AS resistance_mechanism,
       p.verification_status AS verification_status
"""

with get_driver() as driver:
    with driver.session() as session:
        result = session.run(query)
        records = [dict(record) for record in result]

total_cases = len(records)
if total_cases == 0:
    print("⚠️ 数据库中没有病例记录，请先导入数据。")
    exit()

missing_stats = {field: 0 for field in required_fields}

# 统一统计所有必填字段的缺失情况
for record in records:
    for field in required_fields:
        value = record.get(field)
        if value is None or str(value).strip() == '':
            missing_stats[field] += 1

# 计算填充率
total_required_values = len(required_fields) * total_cases
total_missing = sum(missing_stats.values())
fill_rate = ((total_required_values - total_missing) / total_required_values) * 100

print("\n📋 V1 验证：必填字段填充率")
print("=" * 60)
for field, count in missing_stats.items():
    if count > 0:
        print(f"   ⚠️ {field}: {count} 个缺失")
    else:
        print(f"   ✅ {field}: 无缺失")

print(f"\n📊 总填充率: {fill_rate:.1f}%")
print(f"📊 缺失字段总数: {total_missing} / {total_required_values}")

if fill_rate >= 90:
    print("🎉 V1 验证通过！填充率 ≥ 90%")
else:
    print(f"⚠️ V1 验证未通过（{fill_rate:.1f}% < 90%），请检查数据完整性")


📋 V1 验证：必填字段填充率
   ✅ case_id: 无缺失
   ✅ pathogen_id: 无缺失
   ✅ species: 无缺失
   ✅ resistance_mechanism: 无缺失
   ✅ infection_type: 无缺失
   ✅ infection_site: 无缺失
   ✅ specimen_type: 无缺失
   ✅ verification_status: 无缺失

📊 总填充率: 100.0%
📊 缺失字段总数: 0 / 120
🎉 V1 验证通过！填充率 ≥ 90%


In [6]:
driver.close()
print("🔒 数据库连接已关闭")

🔒 数据库连接已关闭
